In [5]:
import os
from typing import List, Dict, Any
import pandas as pd


In [10]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Set up Completed!")

/Users/simransingh/Desktop/RAG Projects/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set up Completed!


In [12]:
doc = Document(
    page_content= "This is a sample document for testing the text splitting functionality. It contains multiple sentences and paragraphs to ensure that the text splitter can handle various cases effectively. The goal is to split this document into smaller chunks that can be processed by language models without losing context or meaning.",
    metadata= {
        "source": "example.txt",
        "page": 1,
        "author": "John Doe",
        "date_created": "2026-05-11",
        "custom_field": "Custom Value"

    }
)

print("Content of the Document:")
print(f"Page Content: {doc.page_content}")
print(f"Metadata {doc.metadata}")
type(doc)

Content of the Document:
Page Content: This is a sample document for testing the text splitting functionality. It contains multiple sentences and paragraphs to ensure that the text splitter can handle various cases effectively. The goal is to split this document into smaller chunks that can be processed by language models without losing context or meaning.
Metadata {'source': 'example.txt', 'page': 1, 'author': 'John Doe', 'date_created': '2026-05-11', 'custom_field': 'Custom Value'}


langchain_core.documents.base.Document

In [22]:
### Text files Reading
os.makedirs("data/text_files", exist_ok=True)
sample_texts = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
"data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
""",
}
for file_path, content in sample_texts.items():
    with open(file_path, "w") as f:
        f.write(content)
print("Sample text files created successfully.")

Sample text files created successfully.


In [26]:
from langchain_community.document_loaders import TextLoader

documents = TextLoader("data/text_files/python_intro.txt", encoding="utf-8").load()

print(type(documents))
print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:100])



<class 'list'>
1
{'source': 'data/text_files/python_intro.txt'}
Python Programming Introduction

Python is a high-level, interpreted programming language known for 


In [34]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader("data/text_files", glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'}, show_progress=True).load()
print(f"Total documents loaded: {len(dir_loader)}")
for i, doc in enumerate(dir_loader):
    print(f"First document metadata: {dir_loader[i].metadata}")
    print(f"First document content preview: {dir_loader[i].page_content[:100]}")


100%|██████████| 2/2 [00:00<00:00, 618.63it/s]

Total documents loaded: 2
First document metadata: {'source': 'data/text_files/python_intro.txt'}
First document content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for 
First document metadata: {'source': 'data/text_files/machine_learning.txt'}
First document content preview: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system


In [38]:
from langchain_text_splitters import (
    CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
)
text = dir_loader[0].page_content
text_splitter = CharacterTextSplitter(
    separator='\n',
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)
chunks = text_splitter.split_text(text)
print(f"Total chunks created: {len(chunks)}")
print(f"First chunk preview: {chunks[0]}")
print('----------')
print(f"Second chunk preview: {chunks[1]}")
print('----------')
print(f"Third chunk preview: {chunks[2]}")


Total chunks created: 3
First chunk preview: Python Programming Introduction
Python is a high-level, interpreted programming language known for its simplicity and readability.
----------
Second chunk preview: Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.
Key Features:
- Easy to learn and use
- Extensive standard library
----------
Third chunk preview: - Cross-platform compatibility
- Strong community support
Python is widely used in web development, data science, artificial intelligence, and automation.


In [50]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(type(recursive_chunks))
print(f"Total chunks created with RecursiveCharacterTextSplitter: {len(recursive_chunks)}")
for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i+1} preview: {chunk}")

<class 'list'>
Total chunks created with RecursiveCharacterTextSplitter: 8
Chunk 1 preview: Python Programming Introduction

Python is a high-level, interpreted programming
Chunk 2 preview: programming language known for its simplicity and readability.
Created by Guido
Chunk 3 preview: by Guido van Rossum and first released in 1991, Python has become one of the
Chunk 4 preview: become one of the most popular
programming languages in the world.

Key
Chunk 5 preview: in the world.

Key Features:
- Easy to learn and use
- Extensive standard
Chunk 6 preview: Extensive standard library
- Cross-platform compatibility
- Strong community
Chunk 7 preview: Strong community support

Python is widely used in web development, data
Chunk 8 preview: development, data science, artificial intelligence, and automation.


In [49]:
token_splitter = TokenTextSplitter(
    chunk_size=40,
    chunk_overlap=20
)
token_chunks = token_splitter.split_text(text)
print(f"Total chunks created with Token TextSplitter: {len(token_chunks)}")
for i, chunk in enumerate(token_chunks):
    print(f"Chunk {i+1} preview: {chunk}")

Total chunks created with Token TextSplitter: 5
Chunk 1 preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become
Chunk 2 preview:  readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
-
Chunk 3 preview:  one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong
Chunk 4 preview:  Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation
Chunk 5 preview:  community support

Python is widely used in web development, data science, artificial intelligence, and automation.
